# VisDrone Human and Car Detection - Colab Training

Run this notebook in Google Colab with a GPU runtime. It clones the project, downloads the Kaggle dataset, trains YOLO on GPU, evaluates the model, and creates counting visualizations.

## 1. Clone This Project

Before running, push this folder to GitHub and paste the repository URL below.

In [ ]:
from pathlib import Path
import os
import shutil

REPO_URL = "https://github.com/Tanvir-kabir-Rahib/VisDrone_Dataset.git"  # change if you fork/rename the repo
PROJECT_ROOT = Path("/content/VisDrone_Dataset")

if not (PROJECT_ROOT / "scripts" / "train_yolo.py").exists():
    assert REPO_URL.startswith("https://github.com/"), "Set REPO_URL to your GitHub repository first."
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    !git clone "$REPO_URL" "$PROJECT_ROOT"

os.chdir(PROJECT_ROOT)
print("Project root:", Path.cwd())

## 2. Install Dependencies and Check GPU

In [ ]:
!pip -q install -r requirements.txt kaggle

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU found. In Colab, use Runtime > Change runtime type > T4 GPU.")

## 3. Mount Google Drive

Training outputs are saved to Drive so they survive Colab disconnects.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/visdrone_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT_STR = str(OUTPUT_ROOT)
print("Output root:", OUTPUT_ROOT)

## 4. Download VisDrone from Kaggle

When prompted, upload your `kaggle.json` API key from Kaggle account settings.

In [ ]:
from google.colab import files

kaggle_json = Path("/root/.kaggle/kaggle.json")
if not kaggle_json.exists():
    print("Upload kaggle.json now.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("kaggle.json was not uploaded.")
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy("kaggle.json", kaggle_json)
    kaggle_json.chmod(0o600)

DATA_DOWNLOAD_ROOT = Path("/content/data")
DATA_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)

if not list(DATA_DOWNLOAD_ROOT.rglob("VisDrone2019-DET-train")):
    !kaggle datasets download -d banuprasadb/visdrone-dataset -p /content/data --unzip

train_dirs = list(DATA_DOWNLOAD_ROOT.rglob("VisDrone2019-DET-train"))
assert train_dirs, "Could not find VisDrone2019-DET-train after download."
DATASET_ROOT = train_dirs[0].parent
DATASET_ROOT_STR = str(DATASET_ROOT)
print("Dataset root:", DATASET_ROOT)

## 5. Validate Dataset and Create Dataset Visualizations

In [ ]:
!python scripts/train_yolo.py --dataset-root "$DATASET_ROOT_STR" --dry-run
!python scripts/analyze_dataset.py --dataset-root "$DATASET_ROOT_STR" --output-dir "$OUTPUT_ROOT_STR/dataset" --sample-limit 6 --size-sample-limit 500

## 6. Optional 1-Epoch Smoke Test

Run this first if you want to confirm GPU training before starting the full 80 epochs.

In [ ]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    !python scripts/train_yolo.py \
      --dataset-root "$DATASET_ROOT_STR" \
      --model yolo11n.pt \
      --epochs 1 \
      --imgsz 640 \
      --batch 8 \
      --device 0 \
      --workers 8 \
      --project "$OUTPUT_ROOT_STR/training" \
      --name smoke-test

## 7. Train YOLO on GPU

The default below is the fast Colab preset: smaller YOLO model, `imgsz=640`, larger batch, more dataloader workers, and disk caching. For better accuracy later, change `MODEL` to `yolo11s.pt` and `IMGSZ` to `960`.

In [ ]:
MODEL = "yolo11n.pt"
EPOCHS = 50
IMGSZ = 640
BATCH = 16
WORKERS = 8
CACHE = "disk"
FREEZE = 10
RUN_NAME = f"visdrone-{MODEL.replace('.pt', '')}-{IMGSZ}"

!python scripts/train_yolo.py \
  --dataset-root "$DATASET_ROOT_STR" \
  --model "$MODEL" \
  --epochs "$EPOCHS" \
  --imgsz "$IMGSZ" \
  --batch "$BATCH" \
  --device 0 \
  --workers "$WORKERS" \
  --cache "$CACHE" \
  --freeze "$FREEZE" \
  --project "$OUTPUT_ROOT_STR/training" \
  --name "$RUN_NAME"

## 8. Locate Best Weights

In [ ]:
best_weights = sorted((OUTPUT_ROOT / "training").rglob("best.pt"), key=lambda p: p.stat().st_mtime)[-1]
BEST_WEIGHTS = str(best_weights)
print("Best weights:", BEST_WEIGHTS)

## 9. Evaluate on Validation Split

In [ ]:
!python scripts/evaluate_yolo.py \
  --dataset-root "$DATASET_ROOT_STR" \
  --weights "$BEST_WEIGHTS" \
  --split val \
  --imgsz "$IMGSZ"

## 10. Create Human/Car Counting Outputs

In [ ]:
!python scripts/detect_count.py \
  --dataset-root "$DATASET_ROOT_STR" \
  --weights "$BEST_WEIGHTS" \
  --split val \
  --limit 12 \
  --output-dir "$OUTPUT_ROOT_STR/predictions"

## 11. Preview Results

In [ ]:
from IPython.display import Image, display

prediction_images = sorted((OUTPUT_ROOT / "predictions").glob("*_count.jpg"))[:4]
for image_path in prediction_images:
    print(image_path.name)
    display(Image(filename=str(image_path), width=900))

print("Count summary:", OUTPUT_ROOT / "predictions" / "count_summary.csv")

## 12. Package Outputs for Submission

In [ ]:
!python scripts/make_demo_slideshow.py --output "$OUTPUT_ROOT_STR/demo/visdrone_demo.gif"
!cd "$OUTPUT_ROOT_STR" && zip -r visdrone_submission_outputs.zip dataset predictions demo training/*/weights/best.pt
print("Submission zip:", OUTPUT_ROOT / "visdrone_submission_outputs.zip")